# Surgical Instrument Detection and Tracking using YOLO11 and ByteTrack

## Dataset Preparation

This notebook prepares the hSDB-Instrument dataset for YOLO11 training.

In [ ]:
import os
import json
import shutil
from pathlib import Path
from ultralytics import YOLO
import torch
import pandas as pd
from IPython.display import Image, display
import cv2
from collections import defaultdict
from collections import Counter


In [ ]:
# Dataset paths
BASE_DIR = Path(r"..\data")

CHOLE_DIR = BASE_DIR /"chole_real" 
GASTRIC_DIR = BASE_DIR /"gastric_real"

# 1. Load COCO Annotations

Load the COCO annotation files for the Chole and Gastric datasets.

In [ ]:
# COCO annotation files
CHOLE_TRAIN_JSON = CHOLE_DIR /"chole_real_train.json"
CHOLE_VAL_JSON = CHOLE_DIR / "chole_real_val.json"

GASTRIC_TRAIN_JSON = GASTRIC_DIR / "gastric_real_train.json"
GASTRIC_VAL_JSON = GASTRIC_DIR / "gastric_real_val.json"

In [ ]:
def load_json(json_path):
    with open(json_path, "r") as f:
        return json.load(f)

In [ ]:
chole_train = load_json(CHOLE_TRAIN_JSON)
chole_val = load_json(CHOLE_VAL_JSON)

gastric_train = load_json(GASTRIC_TRAIN_JSON)
gastric_val = load_json(GASTRIC_VAL_JSON)

In [ ]:
print("Chole Train:")
print(f"Images      : {len(chole_train['images'])}")
print(f"Annotations : {len(chole_train['annotations'])}")
print(f"Categories  : {len(chole_train['categories'])}")

print("\nGastric Train:")
print(f"Images      : {len(gastric_train['images'])}")
print(f"Annotations : {len(gastric_train['annotations'])}")
print(f"Categories  : {len(gastric_train['categories'])}")

# 2. Explore Dataset Classes

This section extracts all instrument classes from the COCO annotation files and displays their corresponding category IDs.

In [ ]:
def get_categories(coco_data):
    return {
        category["id"]: category["name"]
        for category in coco_data["categories"]
    }

In [ ]:
chole_categories = get_categories(chole_train)
gastric_categories = get_categories(gastric_train)

In [ ]:
print("Chole Dataset Classes")
print("-" * 40)

for class_id, class_name in sorted(chole_categories.items()):
    print(f"{class_id:2d} : {class_name}")

In [ ]:
print("Gastric Dataset Classes")
print("-" * 40)

for class_id, class_name in sorted(gastric_categories.items()):
    print(f"{class_id:2d} : {class_name}")

# 3. Merge Instrument Components

This section merges different components of the same surgical instrument (e.g., Head, Body, and Wrist) into a single instrument class to simplify the annotations for instrument-level detection and tracking.

In [ ]:
def merge_class_name(class_name):
    suffixes = [
        "_head",
        "_body",
        "_wrist",
        "_clip",
        "_tip",
        "_Head",
        "_Body",
        "_Wrist",
        "_Clip",
        "_Tip"
    ]
    for suffix in suffixes:
        if class_name.endswith(suffix):
            return class_name.replace(suffix, "")
    return class_name

# 4. Convert COCO Annotations to YOLO Format
This section converts the original COCO annotations into YOLO label files by merging instrument components, assigning class IDs, and normalizing bounding box coordinates.

In [ ]:
def convert_coco_to_yolo_hsdb(
    coco_data,
    output_dir,
    class_to_id,
    prefix=""
):
    os.makedirs(output_dir, exist_ok=True)

    images = {
        img["id"]: img
        for img in coco_data["images"]
    }

    category_id_to_name = {
        cat["id"]: cat["name"]
        for cat in coco_data["categories"]
    }

    annotations_by_image = defaultdict(list)

    for ann in coco_data["annotations"]:
        annotations_by_image[ann["image_id"]].append(ann)

    converted_count = 0

    # IMPORTANT:
    # Loop through ALL images, not only annotated images
    for image_id, img_info in images.items():

        image_name = img_info["file_name"]

        base_name = os.path.splitext(
            os.path.basename(image_name)
        )[0]

        label_name = prefix + base_name + ".txt"

        label_path = os.path.join(
            output_dir,
            label_name
        )

        img_width = img_info["width"]
        img_height = img_info["height"]

        anns = annotations_by_image.get(image_id, [])

        # This creates an empty .txt file if there are no annotations
        with open(label_path, "w") as f:

            for ann in anns:

                category_name = category_id_to_name[
                    ann["category_id"]
                ]

                merged_name = merge_class_name(
                    category_name
                )

                if merged_name not in class_to_id:
                    continue

                class_id = class_to_id[merged_name]

                x, y, w, h = ann["bbox"]

                x_center = (
                    x + w / 2
                ) / img_width

                y_center = (
                    y + h / 2
                ) / img_height

                width = w / img_width
                height = h / img_height

                f.write(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{width:.6f} "
                    f"{height:.6f}\n"
                )

        converted_count += 1

    print(
        f"Converted {converted_count} images to YOLO format."
    )

In [ ]:
DATASET_DIR = r"..\data\hsdb_yolo"

In [ ]:
train_labels_dir = os.path.join(
    DATASET_DIR, "labels", "train"
)

val_labels_dir = os.path.join(
    DATASET_DIR, "labels", "val"
)

os.makedirs(train_labels_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)

print("Labels folders created.")

In [ ]:
counter = Counter()

# Counting on Chole
for ann in chole_train["annotations"]:
    name = merge_class_name(
        chole_categories[ann["category_id"]]
    )
    counter[name] += 1

# Counting on Gastric
for ann in gastric_train["annotations"]:
    name = merge_class_name(
        gastric_categories[ann["category_id"]]
    )
    counter[name] += 1

In [ ]:
merged_classes = sorted(counter.keys())

print("Number of classes:", len(merged_classes))

for i, name in enumerate(merged_classes):
    print(f"{i:2d} : {name}")

# NOTE:
Ligasure was excluded from the final class set because it appears in the validation annotations but has no corresponding training instances. Including it as a separate class would prevent the model from learning that class during training.

In [ ]:
class_to_id = {
    class_name: idx
    for idx, class_name in enumerate(merged_classes)
}

print("Number of classes:", len(class_to_id))

for name, idx in class_to_id.items():
    print(f"{idx:2d} : {name}")

In [ ]:
# Chole
convert_coco_to_yolo_hsdb(
    chole_train,
    train_labels_dir,
    class_to_id,
    prefix="chole_"
)

convert_coco_to_yolo_hsdb(
    chole_val,
    val_labels_dir,
    class_to_id,
    prefix="chole_"
)

# Gastric
convert_coco_to_yolo_hsdb(
    gastric_train,
    train_labels_dir,
    class_to_id,
    prefix="gastric_"
)

convert_coco_to_yolo_hsdb(
    gastric_val,
    val_labels_dir,
    class_to_id,
    prefix="gastric_"
)

In [ ]:
train_label_files = [
    f for f in os.listdir(train_labels_dir)
    if f.endswith(".txt")
]

val_label_files = [
    f for f in os.listdir(val_labels_dir)
    if f.endswith(".txt")
]

print("Train label files:", len(train_label_files))
print("Validation label files:", len(val_label_files))

print("\nExpected train:", 14491 + 29644)
print("Expected validation:", 3573 + 5932)

# 5. Create YOLO Dataset Structure
In this section, the COCO-to-YOLO converted annotations are organized into the standard YOLO directory structure.
The final dataset structure will be:

In [ ]:
# YOLO dataset root directory
YOLO_DIR = Path(r"..\data\hsdb_yolo")

# Image directories
IMAGE_TRAIN_DIR = os.path.join(
    YOLO_DIR, "images", "train"
)

IMAGE_VAL_DIR = os.path.join(
    YOLO_DIR, "images", "val"
)

# Label directories
LABEL_TRAIN_DIR = os.path.join(
    YOLO_DIR, "labels", "train"
)

LABEL_VAL_DIR = os.path.join(
    YOLO_DIR, "labels", "val"
)


# Create directories
for directory in [
    IMAGE_TRAIN_DIR,
    IMAGE_VAL_DIR,
    LABEL_TRAIN_DIR,
    LABEL_VAL_DIR
]:
    os.makedirs(
        directory,
        exist_ok=True
    )


print("YOLO dataset folders created successfully.")

In [ ]:
def copy_images_with_prefix(
    coco_data,
    source_dir,
    destination_dir,
    prefix
):
    os.makedirs(destination_dir, exist_ok=True)

    copied = 0

    for img in coco_data["images"]:

        image_name = os.path.basename(img["file_name"])

        src = source_dir / img["file_name"]

        new_name = prefix + image_name

        dst = os.path.join(
            destination_dir,
            new_name
        )

        if src.exists():
            shutil.copy2(src, dst)
            copied += 1

    print(
        f"Copied {copied} images to {destination_dir}"
    )

In [ ]:
# Source image directories
CHOLE_TRAIN_IMAGE_DIR = CHOLE_DIR / "chole_real_train"
CHOLE_VAL_IMAGE_DIR = CHOLE_DIR / "chole_real_val"

GASTRIC_TRAIN_IMAGE_DIR = GASTRIC_DIR / "gastric_real_train"
GASTRIC_VAL_IMAGE_DIR = GASTRIC_DIR / "gastric_real_val"

In [ ]:
# TRAIN

copy_images_with_prefix(
    chole_train,
    CHOLE_TRAIN_IMAGE_DIR,
    IMAGE_TRAIN_DIR,
    "chole_"
)

copy_images_with_prefix(
    gastric_train,
    GASTRIC_TRAIN_IMAGE_DIR,
    IMAGE_TRAIN_DIR,
    "gastric_"
)


# VALIDATION

copy_images_with_prefix(
    chole_val,
    CHOLE_VAL_IMAGE_DIR,
    IMAGE_VAL_DIR,
    "chole_"
)

copy_images_with_prefix(
    gastric_val,
    GASTRIC_VAL_IMAGE_DIR,
    IMAGE_VAL_DIR,
    "gastric_"
)

In [ ]:
print("Training images :", len(os.listdir(IMAGE_TRAIN_DIR)))
print("Training labels :", len(os.listdir(LABEL_TRAIN_DIR)))
print()

print("Validation images :", len(os.listdir(IMAGE_VAL_DIR)))
print("Validation labels :", len(os.listdir(LABEL_VAL_DIR)))

In [ ]:
def check_image_label_matching(image_dir, label_dir):

    image_basenames = {
        os.path.splitext(f)[0]
        for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    }

    label_basenames = {
        os.path.splitext(f)[0]
        for f in os.listdir(label_dir)
        if f.endswith(".txt")
    }

    missing_labels = image_basenames - label_basenames
    orphan_labels = label_basenames - image_basenames

    print("Images:", len(image_basenames))
    print("Labels:", len(label_basenames))
    print("Missing labels:", len(missing_labels))
    print("Orphan labels:", len(orphan_labels))

    if missing_labels:
        print("\nExample missing labels:")
        print(list(missing_labels)[:10])

    if orphan_labels:
        print("\nExample orphan labels:")
        print(list(orphan_labels)[:10])


print("TRAIN")
print("-" * 40)
check_image_label_matching(
    IMAGE_TRAIN_DIR,
    LABEL_TRAIN_DIR
)

print("\nVALIDATION")
print("-" * 40)
check_image_label_matching(
    IMAGE_VAL_DIR,
    LABEL_VAL_DIR
)

# 6. Validate YOLO Annotations

In this section, the generated YOLO annotation files are validated to ensure that the dataset is correctly prepared for training.

The validation process checks:

- Correct YOLO annotation format.
- Valid class IDs within the defined class range.
- Normalized bounding box coordinates.
- Absence of corrupted or invalid label files.

These checks help ensure that the dataset is ready for the YOLO training stage.

In [ ]:
def check_yolo_labels(label_dir, num_classes):
    errors = []
    total = 0

    for file in os.listdir(label_dir):
        if not file.endswith(".txt"):
            continue

        path = os.path.join(label_dir, file)

        with open(path, "r") as f:
            lines = f.readlines()

        for line in lines:
            total += 1
            values = line.strip().split()

            if len(values) != 5:
                errors.append((file, "wrong format", values))
                continue

            cls, x, y, w, h = map(float, values)

            if cls < 0 or cls >= num_classes:
                errors.append((file, "wrong class", cls))

            if not all(0 <= v <= 1 for v in [x, y, w, h]):
                errors.append((file, "bbox out of range", values))

    return total, errors


train_count, train_errors = check_yolo_labels(
    LABEL_TRAIN_DIR,
    num_classes=20
)

val_count, val_errors = check_yolo_labels(
    LABEL_VAL_DIR,
    num_classes=20
)

print("Train annotations:", train_count)
print("Train errors:", len(train_errors))

print()

print("Validation annotations:", val_count)
print("Validation errors:", len(val_errors))

# 7. Create YOLO Configuration File

In this section, the YOLO dataset configuration file (`data.yaml`) is created.

This file defines:

- Training and validation image paths.
- Number of object classes.
- Class names used during training.

The configuration file will be used by the YOLO model during the training process.

In [ ]:
import yaml

data_yaml = {
    "path": DATASET_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 20,
    "names": list(class_to_id.keys())
}

yaml_path = os.path.join(DATASET_DIR, "data.yaml")

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Created:", yaml_path)

with open(yaml_path, "r") as f:
    print(f.read())


# 8. Install and Load YOLO11

In this section, the Ultralytics YOLO11 library is installed and imported.

A pretrained YOLO11 model is then loaded to prepare for training on the custom surgical instrument detection dataset.

The training process will use the dataset configuration file (`data.yaml`) created in the previous section.

In [ ]:
%pip install -q ultralytics

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
model = YOLO("yolo11s.pt")

print(model)

# 9. Train YOLO11 Model

In this section, the YOLO11 model is trained on the custom surgical instrument detection dataset.

The training process uses the dataset configuration file (`data.yaml`), pretrained weights, and selected hyperparameters.

Early stopping is enabled to automatically stop training if the validation performance does not improve for a specified number of epochs.

In [ ]:
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    workers=4,
    device=0,
    project="runs",
    name="yolo11s_hsdb",
    pretrained=True,
    save=True,
    verbose=True
)

# 10. Analyze Training Results

In this section, the training results are analyzed to evaluate model performance.

The generated metrics, loss curves, and validation results are inspected to understand the learning behavior of the YOLO11 model.

In [ ]:
results_dir = r"..\data\yolo11s_hsdb"

print(os.listdir(results_dir))

In [ ]:
weights_dir = os.path.join(results_dir, "weights")

print(os.listdir(weights_dir))

In [ ]:
results_image = os.path.join(results_dir, "results.png")

display(Image(filename=results_image))

In [ ]:
results_csv = os.path.join(results_dir, "results.csv")

results_df = pd.read_csv(results_csv)

results_df.head()

In [ ]:
output_dir = r"..\results"

os.makedirs(output_dir, exist_ok=True)

shutil.copy(
    os.path.join(results_dir, "results.png"),
    os.path.join(output_dir, "training_results.png")
)

shutil.copy(
    results_csv,
    os.path.join(output_dir, "training_results.csv")
)

print("Training results saved.")

In [ ]:
best_model_path = os.path.join(weights_dir, "best.pt")

print("Best model:", best_model_path)

# 11. Model Evaluation and Tracking

In this section, the trained YOLO11 model is evaluated using the best saved weights.

The model performance is measured on the validation dataset, and the detection outputs are prepared for the tracking stage using ByteTrack.

In [ ]:
best_model_path = os.path.join(
    weights_dir,
    "best.pt"
)

model = YOLO(best_model_path)

print("Best model loaded.")

In [ ]:
eval_dir = r"..\results"

os.makedirs(eval_dir, exist_ok=True)

In [ ]:
metrics = model.val(
    data=yaml_path,
    imgsz=640,
    batch=16,
    device=0
)

In [ ]:
# ================================
# Per-Class Detection Metrics
# ================================

names = model.names

try:
    class_metrics = metrics.box

    per_class_df = pd.DataFrame({
        "Class": [names[i] for i in range(len(names))],
        "Precision": class_metrics.p,
        "Recall": class_metrics.r,
        "mAP50": class_metrics.ap50,
        "mAP50-95": class_metrics.ap
    })

    display(per_class_df)

    save_path = os.path.join(eval_dir, "per_class_metrics.csv")
    per_class_df.to_csv(save_path, index=False)

    print(f"Saved to: {save_path}")

except Exception as e:
    print(e)
    print("Per-class metrics are not available in this Ultralytics version.")

In [ ]:
metrics_file = os.path.join(
    eval_dir,
    "validation_metrics.txt"
)

with open(metrics_file, "w") as f:
    f.write(f"Precision: {metrics.box.mp:.4f}\n")
    f.write(f"Recall: {metrics.box.mr:.4f}\n")
    f.write(f"mAP50: {metrics.box.map50:.4f}\n")
    f.write(f"mAP50-95: {metrics.box.map:.4f}\n")

print("Metrics saved:", metrics_file)

In [ ]:
validation_dir = metrics.save_dir

validation_files = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png"
]

for file in validation_files:
    source = os.path.join(validation_dir, file)
    destination = os.path.join(eval_dir, file)

    if os.path.exists(source):
        shutil.copy(source, destination)

print("Validation plots saved.")

In [ ]:
prediction_results = model.predict(
    source=IMAGE_VAL_DIR,
    imgsz=640,
    conf=0.25,
    save=True,
    project=eval_dir,
    name="predictions"
)

print("Predictions saved.")

## 12. ByteTrack Integration

In this section, the YOLO11 detection outputs are combined with ByteTrack to perform multi-object tracking.

The detected surgical instruments are assigned unique IDs and tracked across video frames.

In [ ]:
# Load Input Video

video_path = r"..\data\path_to_video.mp4"

tracking_dir = os.path.join(
    eval_dir,
    "tracking"
)

os.makedirs(tracking_dir, exist_ok=True)

In [ ]:
# Run YOLO + ByteTrack Tracking

tracking_results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    imgsz=640,
    conf=0.35,
    save=True,
    project=tracking_dir,
    name="tracked_video",
    persist=True
)

print("Tracking completed.")

In [ ]:
tracked_video_path = r"..\results\tracking_video.mp4"
print(tracked_video_path)

In [ ]:
# ================================
# Tracking Statistics
# ================================

track_lengths = defaultdict(int)

for result in tracking_results:

    if result.boxes.id is None:
        continue

    ids = result.boxes.id.cpu().numpy().astype(int)

    for tid in ids:
        track_lengths[tid] += 1

num_tracks = len(track_lengths)

print("="*40)
print("Tracking Statistics")
print("="*40)
print(f"Total Tracks : {num_tracks}")

if num_tracks > 0:

    lengths = list(track_lengths.values())

    print(f"Average Track Length : {sum(lengths)/len(lengths):.2f} frames")
    print(f"Longest Track        : {max(lengths)} frames")
    print(f"Shortest Track       : {min(lengths)} frames")

In [ ]:
# Check Tracking Output

for root, dirs, files in os.walk(tracking_dir):
    for file in files:
        if file.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            print(os.path.join(root, file))

In [ ]:
tracking_output_dir = os.path.join(
    tracking_dir,
    "tracked_video"
)

for root, dirs, files in os.walk(tracking_output_dir):
    for file in files:
        if file.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            tracked_video_path = os.path.join(root, file)
            print("Tracked video:", tracked_video_path)

In [ ]:
sample_dir = os.path.join(
    eval_dir,
    "tracking_samples"
)

os.makedirs(sample_dir, exist_ok=True)

cap = cv2.VideoCapture(tracked_video_path)

frame_numbers = [50, 1500, 3000]

for frame_number in frame_numbers:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

    ret, frame = cap.read()

    if ret:
        output_path = os.path.join(
            sample_dir,
            f"frame_{frame_number}.jpg"
        )

        cv2.imwrite(output_path, frame)

cap.release()

print("Tracking samples saved.")